# GUTS on YJMob-14d

The Full-Graph baseline, a GUTS source run and shard unlearning of a node forget set on YJMob-14d (K=130, seed 2026).


## Settings

Paths, GPUs, seed, forget set and run directories.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "guts").is_dir() and (REPO_ROOT / "configs").is_dir(), \
    "Start Jupyter from the repository root."
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from guts.method.utils import timestamp_id
from guts.yjmob import full_graph, pipeline
from guts.yjmob.data import preprocess
from guts.yjmob.unlearning import forget_sets, unlearn_batch, unlearn_batch_retry

In [2]:
NUM_GPUS = 4                         # GPUs for shard training and evaluation
SEED = 2026                          # run seed (partition and shard training)
CONFIG = "configs/yjmob_14d.yaml"    # K=130 and the other settings of the paper

DATASET_NAME = "dataset1"            # YJMob CSV under processed/yjmob/raw/
DATA_DIR = "processed/yjmob_1_14d"   # days 0-13
STRATA = "guts/yjmob/unlearning/strata.json"

FORGET_TYPE = "node"
FORGET_PERCENT = 0.05                # % of the train cells
FORGET_SET = f"processed/forget_sets/yjmob/Node/forget_node_seed{SEED}_{FORGET_PERCENT:g}pct.json"
CLEAN_DIR = f"processed/yjmob_1_14d_clean_node_seed{SEED}_{FORGET_PERCENT:g}pct"

RUN_ID = timestamp_id()              # new runs each time this cell runs
BASE_RUN_DIR = REPO_ROOT / "Results/runs/yjmob_14d" / f"{RUN_ID}_full_graph_seed{SEED}"
SHARD_RUN_DIR = REPO_ROOT / "Results/shard-runs/yjmob_14d" / f"{RUN_ID}_guts_k130_seed{SEED}"
UNLEARN_RUN_DIR = (REPO_ROOT / "Results/shard-unlearns/yjmob_14d"
                   / f"{RUN_ID}_guts_k130_seed{SEED}_{FORGET_TYPE}_{FORGET_PERCENT:g}pct")
for run_dir in (BASE_RUN_DIR, SHARD_RUN_DIR, UNLEARN_RUN_DIR):
    print("run directory:", run_dir.relative_to(REPO_ROOT))

run directory: Results/runs/yjmob_14d/261027_15h51m07s_full_graph_seed2026
run directory: Results/shard-runs/yjmob_14d/261027_15h51m07s_guts_k130_seed2026
run directory: Results/shard-unlearns/yjmob_14d/261027_15h51m07s_guts_k130_seed2026_node_0.05pct


## Data

The 14-day YJMob dataset, the quartile strata file, the node forget set and the clean data without it.

In [3]:
if (REPO_ROOT / DATA_DIR / "next_location_meta.json").exists():
    print("dataset exists:", DATA_DIR)
else:
    preprocess.main(["--dataset_name", DATASET_NAME, "--day_range", "0", "13",
                     "--out_name", Path(DATA_DIR).name])

if (REPO_ROOT / STRATA).exists():
    print("strata file exists:", STRATA)
else:
    forget_sets.main(["strata", "--source_dir", DATA_DIR])

dataset exists: processed/yjmob_1_14d
strata file exists: guts/yjmob/unlearning/strata.json


In [4]:
if (REPO_ROOT / STRATA).exists():
    print("strata file exists:", STRATA)
else:
    forget_sets.main(["strata", "--source_dir", DATA_DIR])

if (REPO_ROOT / FORGET_SET).exists():
    print("forget set exists:", FORGET_SET)
else:
    forget_sets.main(["batch", "--source_dir", DATA_DIR, "--percent", f"{FORGET_PERCENT:g}",
                      "--types", FORGET_TYPE, "--seed", str(SEED)])

if (REPO_ROOT / CLEAN_DIR / "next_location_meta.json").exists():
    print("clean data exists:", CLEAN_DIR)
else:
    preprocess.main(["--dataset_name", DATASET_NAME, "--day_range", "0", "13",
                     "--fixed_grid_info_from", f"{DATA_DIR}/next_location_meta.pt",
                     "--exclude_nodes_from", FORGET_SET, "--out_name", Path(CLEAN_DIR).name])

strata file exists: guts/yjmob/unlearning/strata.json
forget set exists: processed/forget_sets/yjmob/Node/forget_node_seed2026_0.05pct.json
clean data exists: processed/yjmob_1_14d_clean_node_seed2026_0.05pct


## Full-Graph baseline

One GraphSAGE + GRU model trained on the whole graph, evaluated with the shard routing evaluation of a single shard.

In [5]:
if (BASE_RUN_DIR / "results.json").exists():
    print("finished run exists:", BASE_RUN_DIR.relative_to(REPO_ROOT))
else:
    argv = ["--config", CONFIG, "--num-gpus", str(NUM_GPUS), "--seed", str(SEED)]
    if BASE_RUN_DIR.exists():
        argv += ["--resume-dir", str(BASE_RUN_DIR)]
    else:
        argv += ["--out", str(BASE_RUN_DIR.parent), "--override", f"output.run_name={BASE_RUN_DIR.name}"]
    full_graph.main(argv)

[15:51:07] Full-Graph baseline  2026-10-27  run=Results/runs/yjmob_14d/261027_15h51m07s_full_graph_seed2026  seed=2026 (given)  GPUs=4
[15:51:07]   data=processed/yjmob_1_14d  days 0-13 (14)
[15:51:07] STEP 1/3 full-graph data (one shard with every cell)
[15:51:16]   data done  14 day graphs (1,418-652,117 edges)  rolling graphs (window 3, target days 3-13)  9s
[15:51:16] STEP 2/3 training
[15:51:16]   training 1 model on 4 GPUs (torchrun data parallel)
[17:14:12]   training done  4976s
[17:14:12]   best E57/stopped E72  train time 4389s
[17:14:12]     train  MRR 0.3066  ACC@1 0.2136  ACC@5 0.4028  ACC@10 0.4885
[17:14:12]     val    MRR 0.2806  ACC@1 0.1924  ACC@5 0.3710  ACC@10 0.4530
[17:14:12] STEP 3/3 shard routing evaluation (test)
[17:14:44]   test  MRR 0.2480 ACC@1 0.1645 ACC@5 0.3315 ACC@10 0.4131  routable 99.97%  32s
[17:14:44] done  5017s  -> Results/runs/yjmob_14d/261027_15h51m07s_full_graph_seed2026/results.json


## GUTS source run

Per-day GUTS partitioning, one GraphSAGE model per shard, and shard routing evaluation on the test split.

In [6]:
if (SHARD_RUN_DIR / "results.json").exists():
    print("finished run exists:", SHARD_RUN_DIR.relative_to(REPO_ROOT))
else:
    argv = ["--config", CONFIG, "--num-gpus", str(NUM_GPUS), "--seed", str(SEED)]
    if SHARD_RUN_DIR.exists():
        argv += ["--resume-dir", str(SHARD_RUN_DIR)]
    else:
        argv += ["--override", f"output.root={SHARD_RUN_DIR.parent}", f"output.run_name={SHARD_RUN_DIR.name}"]
    pipeline.main(argv)

[17:14:44] GUTS pipeline  2026-10-27  run=Results/shard-runs/yjmob_14d/261027_15h51m07s_guts_k130_seed2026  seed=2026 (given)  K=130  GPUs=4
[17:14:44]   data=processed/yjmob_1_14d  days 0-13 (14)
[17:14:44] STEP 1/3 partition (14 days, K=130)
[17:14:44]   partition fit: 14 days on 8 worker processes
[17:19:40]   partition fit done  296s
[17:20:32]   subgraph assignment done  14/14 days  52s
[17:20:32]   partition done  mean over 14 days: shards/traj 2.602  cstr 0.462  348s
[17:20:32] STEP 2/3 shard data + shard training
[17:20:32]   shard data: building 130 shards
[17:21:27]   sessions split  train 1,117,419  val 166,155  test 319,137  -> 130 shards written
[17:21:57]   shard data done  85s
[17:22:02]   rolling graphs (window 3, target days 3-13) done  5s
[17:22:02]   training 130 shards on 4 GPU(s)
[17:22:02]   shards go to the next free GPU, largest first; lines below are the shards trained on GPU 0
[17:38:11]   shard 129  E35/50   813s  GPU0 #1, all 13/130
               train  MRR

## Unlearning

The node forget set removed from the GUTS run: only the affected shards are rebuilt from the clean data and retrained.

In [7]:
if not (SHARD_RUN_DIR / "results.json").exists():
    raise FileNotFoundError(f"{SHARD_RUN_DIR.relative_to(REPO_ROOT)} not finished: run the GUTS source run cell first")
if (UNLEARN_RUN_DIR / "results_unlearn.json").exists():
    print("finished run exists:", UNLEARN_RUN_DIR.relative_to(REPO_ROOT))
elif (UNLEARN_RUN_DIR / "unlearn_request.json").exists():
    unlearn_batch_retry.main(["--out_dir", str(UNLEARN_RUN_DIR), "--num_gpus", str(NUM_GPUS)])
else:
    unlearn_batch.main(["--state_dir", str(SHARD_RUN_DIR), "--forget_type", FORGET_TYPE,
                        "--forget_set_path", FORGET_SET, "--clean_source_dir", CLEAN_DIR,
                        "--num_gpus", str(NUM_GPUS), "--out_root", str(UNLEARN_RUN_DIR.parent),
                        "--run_name", UNLEARN_RUN_DIR.name])

[18:24:59] GUTS unlearn (node)  2026-10-27  source=Results/shard-runs/yjmob_14d/261027_15h51m07s_guts_k130_seed2026  out=Results/shard-unlearns/yjmob_14d/261027_15h51m07s_guts_k130_seed2026_node_0.05pct  GPUs=4
[18:25:36]   index: 130 shards, 4,033,798 train segments  37s
[18:25:36]   affected shards 108/130  (node: 14 records)
[18:25:37]   rebuilding 108 shards from the clean data (source partitions of days 0-13)
[18:26:33]   sessions split  train 1,117,071  val 166,104  test 319,064  -> 108 shards written
[18:27:03]   rebuild done (shard data + rolling graphs)  85s
[18:27:03]   retraining 108 shards on 4 GPU(s); shards go to the next free GPU, largest first; lines below are the shards trained on GPU 0
[18:43:03]   shard 129  E35/50   807s  GPU0 #1, all 13/108
               train  MRR 0.2331  ACC@1 0.1521  ACC@5 0.3131  ACC@10 0.3940
               val    MRR 0.2071  ACC@1 0.1311  ACC@5 0.2813  ACC@10 0.3576
[18:45:40]   shard  96  E33/48   132s  GPU0 #2, all 14/108
               tr

## Results

Prediction metrics of the three runs on the routable test transitions and the partition quality of the GUTS run
after boundary refinement (mean over the 14 days).

In [10]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

base = load_json(BASE_RUN_DIR / "results.json")
shard = load_json(SHARD_RUN_DIR / "results.json")
unlearn = load_json(UNLEARN_RUN_DIR / "results_unlearn.json")
K = shard["partition"]["num_shards"]

evals = {
    "Full-Graph": base["graphsage_agg_eval"],
    f"GUTS (K={K})": shard["graphsage_agg_eval"],
    f"GUTS unlearn ({FORGET_TYPE} {FORGET_PERCENT:g}%)": unlearn["eval"],
}
prediction = pd.DataFrame([{
    "MRR": ev["routable"]["mrr"], "ACC@1": ev["routable"]["acc@1"],
    "ACC@5": ev["routable"]["acc@5"], "ACC@10": ev["routable"]["acc@10"]
} for ev in evals.values()], index=list(evals))
display(prediction.round(4))

tb, ts, tu = base["timing_ms"], shard["timing_ms"], unlearn["timing_ms"]
timing = pd.Series({
    "Full-Graph: training (s)": tb["train"] / 1000,
    "Full-Graph: wall time (min)": base["wall_time_s"] / 60,
    "GUTS: partition, 14 days (s)": ts["partition"] / 1000,
    "GUTS: train per shard, mean (s)": ts["per_shard_train"]["mean"] / 1000,
    "GUTS: wall time (min)": shard["wall_time_s"] / 60,
    "Unlearn: affected shards": f"{len(unlearn['forget_request']['affected_shards'])}/{K}",
    "Unlearn: retraining (s)": tu["retrain_wall_s"],
    "Unlearn: wall time (min)": unlearn["wall_time_s"] / 60,
}, name="time")
display(timing.to_frame())

,MRR,ACC@1,ACC@5,ACC@10
Full-Graph,0.2480,0.1645,0.3315,0.4131
GUTS (K=130),0.2263,0.1501,0.3025,0.3744
GUTS unlearn (node 0.05%),0.2258,0.1496,0.3020,0.3739


,time
Full-Graph: training (s),4389.051
Full-Graph: wall time (min),83.615
"GUTS: partition, 14 days (s)",295.879
"GUTS: train per shard, mean (s)",87.237462
GUTS: wall time (min),70.241667
Unlearn: affected shards,108/130
Unlearn: retraining (s),3078.1
Unlearn: wall time (min),56.173333
